In [7]:
import os
import pandas as pd
import numpy as np
from google.oauth2.service_account import Credentials
from googleapiclient.discovery import build
from dotenv import load_dotenv
from pycoingecko import CoinGeckoAPI
import warnings

warnings.filterwarnings("ignore")

# Load environment variables
load_dotenv()

# Google Sheets configuration
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly']
SERVICE_ACCOUNT_FILE = os.getenv('SERVICE_ACCOUNT_FILE')
SPREADSHEET_ID = '1156S3pm9vQ2JP5bE0yvpj_Y8FE55ecovHS_JvIys5dc'

# Authenticate and build the service
creds = Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SCOPES)
service = build('sheets', 'v4', credentials=creds)

def get_allocation_data(sheet_name):
    """
    Pulls allocation data from the specified Google Sheets sheet.
    
    Parameters:
        sheet_name (str): The name of the sheet (e.g. "EXC", "HB", or "LC")
    
    Returns:
        pd.DataFrame: DataFrame built from the sheet values (header in first row)
    """
    range_name = f'{sheet_name}!A:AAZ'
    result = service.spreadsheets().values().get(
        spreadsheetId=SPREADSHEET_ID,
        range=range_name
    ).execute()
    values = result.get('values', [])
    if not values:
        print(f'No data found in sheet {sheet_name}.')
        return None
    # Assume the first row is the header (tickers and possibly a "date" column)
    df = pd.DataFrame(values[1:], columns=values[0])
    return df

def parse_allocation_row(df):
    """
    Converts the latest (last) row of the allocation DataFrame to a dictionary of ticker: allocation,
    and extracts the start date.
    """
    allocation_date = None
    allocations = df.iloc[-1]
    allocation_dict = {}
    
    # Look for date column with case-insensitive match (including Portuguese "Data")
    date_column = next((col for col in df.columns if col.lower() in ['date', 'data']), None)
    if date_column:
        try:
            allocation_date = pd.to_datetime(allocations[date_column])
            print(f"Found date: {allocation_date}")
        except Exception as e:
            print(f"Error parsing date from column {date_column}:", e)
    else:
        print("No date column found in the sheet (looked for 'date' or 'data')")
    
    # Process allocations
    for ticker, value in allocations.items():
        if ticker.lower() not in ['date', 'data']:  # Skip date column
            if isinstance(value, str):
                value = value.strip()
                if value.endswith('%'):
                    try:
                        allocation = float(value.replace('%', '')) / 100
                    except ValueError:
                        allocation = 0.0
                else:
                    try:
                        allocation = float(value)
                    except ValueError:
                        allocation = 0.0
            else:
                try:
                    allocation = float(value)
                except Exception:
                    allocation = 0.0
            allocation_dict[ticker.upper()] = allocation
    
    return allocation_dict, allocation_date

def get_latest_allocations():
    """
    Pulls the latest allocation data for the three portfolios.
    The portfolios are defined by sheet names:
      carteira_EXC -> sheet "EXC"
      carteira_HB  -> sheet "HB"
      carteira_LC  -> sheet "LC"
    
    Returns:
         dict: Dictionary with keys 'carteira_EXC', 'carteira_HB', and 'carteira_LC'
               and each value is a tuple (allocation_dict, allocation_date).
    """
    portfolios = {
        'carteira_EXC': 'EXC',
        'carteira_HB': 'HB',
        'carteira_LC': 'LC'
    }
    all_allocations = {}
    for portfolio_key, sheet_name in portfolios.items():
        df = get_allocation_data(sheet_name)
        if df is not None:
            allocation_dict, allocation_date = parse_allocation_row(df)
            all_allocations[portfolio_key] = (allocation_dict, allocation_date)
    return all_allocations

def get_ticker_mapping(portfolio_assets):
    """
    Uses the CoinGecko API to retrieve the coins list and creates a mapping 
    from coin ticker (upper-case) to coin id.
    """
    cg = CoinGeckoAPI()
    coins_list = cg.get_coins_list()
    coins_df = pd.DataFrame(coins_list)
    
    # Define known mappings for ambiguous tickers
    known_mappings = {
        'VIRTUAL': 'virtual-protocol',
        'HYPE': 'hyperliquid',
        'YNE': 'yesnoerror'
    }
    
    # Create mapping by first checking known mappings, then filtering by portfolio_assets
    mapping = {}
    
    # Add known mappings first
    for ticker, coin_id in known_mappings.items():
        if coin_id in portfolio_assets:
            mapping[ticker] = coin_id
    
    # Add remaining mappings from CoinGecko
    filtered_coins_df = coins_df[coins_df['id'].isin(portfolio_assets)]
    for _, row in filtered_coins_df.iterrows():
        ticker = row['symbol'].upper()
        if ticker not in mapping:  # Don't override known mappings
            mapping[ticker] = row['id']
    
    return mapping

def build_dataframe(portfolio_assets, allocation_dict=None):
    """
    Builds a price close DataFrame using either all assets in portfolio_assets list
    or only the tickers with non-zero allocation if allocation_dict is provided.
    """
    mapping = get_ticker_mapping(portfolio_assets)
    price_data = {}
    
    if allocation_dict is not None:
        assets_to_process = {ticker: alloc for ticker, alloc in allocation_dict.items() if alloc > 0}
    else:
        reverse_mapping = {v: k for k, v in mapping.items()}
        assets_to_process = {reverse_mapping.get(asset_id, asset_id): 1 for asset_id in portfolio_assets}
    
    # Process each asset
    for ticker in assets_to_process:
        if ticker in mapping:
            coin_id = mapping[ticker]
            file_path = os.path.join("micro", "assetData", f"{coin_id}.csv")
            if os.path.exists(file_path):
                df = pd.read_csv(file_path)
                if 'date' in df.columns and 'close' in df.columns:
                    df['date'] = pd.to_datetime(df['date'])
                    df = df.drop_duplicates(subset=['date'])  # Remove duplicate dates
                    df.set_index('date', inplace=True)
                    price_data[ticker] = df['close']
                else:
                    print(f"Required columns not found in {file_path}")
            else:
                print(f"File not found: {file_path}")
        else:
            print(f"Ticker {ticker} not found in ticker mapping.")
    
    if not price_data:
        return pd.DataFrame()
    
    # Create DataFrame with all series and ensure index is unique
    price_df = pd.DataFrame(price_data)
    price_df = price_df[~price_df.index.duplicated(keep='first')]  # Keep first occurrence of duplicate indices
    
    # Sort index to ensure chronological order
    price_df.sort_index(inplace=True)
    
    # For each column (asset), fill NaN values with the first available price
    for column in price_df.columns:
        first_valid_price = price_df[column].first_valid_index()
        if first_valid_price is not None:
            price_df[column].fillna(price_df[column][first_valid_price], inplace=True)
    
    return price_df

if __name__ == "__main__":
    # Import the portfolio lists from assetsRoster (only these three are used)
    from scripts.assetsRoster import carteira_EXC, carteira_HB, carteira_LC

    # (1) Pull allocation data from Google Sheets and parse the percentages and allocation date.
    latest_allocations = get_latest_allocations()
    
    exc_alloc, exc_date = latest_allocations.get('carteira_EXC', ({}, None))
    hb_alloc, hb_date = latest_allocations.get('carteira_HB', ({}, None))
    lc_alloc, lc_date = latest_allocations.get('carteira_LC', ({}, None))
    
    # (3) Build a price close DataFrame for each portfolio using only non-zero allocations.
    # Each portfolio's price data will start on the allocation date extracted from the sheet.
    exc_price_df = build_dataframe(carteira_EXC, exc_alloc)
    hb_price_df = build_dataframe(carteira_HB, hb_alloc)
    lc_price_df = build_dataframe(carteira_LC, lc_alloc)
    
    # Print (or further process) the resulting DataFrames
    print("EXC Portfolio Price Data:")
    print(exc_price_df.head())
    
    print("\nHB Portfolio Price Data:")
    print(hb_price_df.head())
    
    print("\nLC Portfolio Price Data:")
    print(lc_price_df.head())

Found date: 2025-02-12 00:00:00
Found date: 2025-02-12 00:00:00
Found date: 2025-02-12 00:00:00
EXC Portfolio Price Data:
               BTC      ETH      LINK       AAVE       UNI       SOL  \
date                                                                   
2013-04-27  135.30  2.83162  0.225377  56.163203  3.443832  0.957606   
2013-04-28  141.96  2.83162  0.225377  56.163203  3.443832  0.957606   
2013-04-29  135.30  2.83162  0.225377  56.163203  3.443832  0.957606   
2013-04-30  117.00  2.83162  0.225377  56.163203  3.443832  0.957606   
2013-05-01  103.43  2.83162  0.225377  56.163203  3.443832  0.957606   

              PENDLE       AKT    RENDER      ONDO      AERO    MORPHO  \
date                                                                     
2013-04-27  1.801588  0.402013  0.051188  0.219553  0.061438  1.268028   
2013-04-28  1.801588  0.402013  0.051188  0.219553  0.061438  1.268028   
2013-04-29  1.801588  0.402013  0.051188  0.219553  0.061438  1.268028   
201

In [8]:
from geckoAPI.assetsRoster import carteira_EXC, carteira_HB, carteira_LC, ROSTER

roster_price_df = build_dataframe(ROSTER)

roster_price_df



,GMX,ETHFI,ALPHA,INJ,FTM,AKT,SHDW,AERO,YNE,SCRT,...,GENE,SRM,VIRTUAL,FARTCOIN,ETC,COMP,POLIS,LUNA,BCH,XTZ
date,,,,,,,,,,,,,,,,,,,,,
2013-04-27,15.732550,3.143911,0.051652,0.767915,0.013985,0.402013,2.203155,0.061438,0.017465,0.452460,...,15.096406,1.570990,0.014423,0.040677,0.752345,78.579316,8.634977,5.151269,767.767713,2.937866
2013-04-28,15.732550,3.143911,0.051652,0.767915,0.013985,0.402013,2.203155,0.061438,0.017465,0.452460,...,15.096406,1.570990,0.014423,0.040677,0.752345,78.579316,8.634977,5.151269,767.767713,2.937866
2013-04-29,15.732550,3.143911,0.051652,0.767915,0.013985,0.402013,2.203155,0.061438,0.017465,0.452460,...,15.096406,1.570990,0.014423,0.040677,0.752345,78.579316,8.634977,5.151269,767.767713,2.937866
2013-04-30,15.732550,3.143911,0.051652,0.767915,0.013985,0.402013,2.203155,0.061438,0.017465,0.452460,...,15.096406,1.570990,0.014423,0.040677,0.752345,78.579316,8.634977,5.151269,767.767713,2.937866
2013-05-01,15.732550,3.143911,0.051652,0.767915,0.013985,0.402013,2.203155,0.061438,0.017465,0.452460,...,15.096406,1.570990,0.014423,0.040677,0.752345,78.579316,8.634977,5.151269,767.767713,2.937866
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-02-08,20.140999,1.137033,0.049780,13.847804,0.415192,1.962304,0.278014,0.834706,0.029051,0.252868,...,0.155292,0.022178,1.248997,0.545826,20.157952,50.832629,0.096842,0.255615,323.426436,0.876994
2025-02-09,23.101139,1.116357,0.049628,13.750636,0.408605,1.901586,0.281246,0.802435,0.022462,0.248497,...,0.153624,0.022131,1.131159,0.485526,20.198741,51.112170,0.102564,0.252627,325.014635,0.881320
2025-02-10,24.167418,1.127893,0.050647,14.541324,0.435580,1.943165,0.296991,0.817332,0.036759,0.260745,...,0.154942,0.022016,1.190039,0.564563,20.642692,54.471806,0.103113,0.264496,328.805400,0.889445


In [34]:
import pandas_ta as ta
import plotly.express as px

def analyze_portfolio_technicals(price_df, start_date=None):
    """
    Calculates current drawdown, max historical drawdown, and RSI for each asset
    and creates a scatter plot of RSI vs Drawdown.
    
    Parameters:
        price_df (pd.DataFrame): DataFrame with price history for portfolio assets
        start_date (str or datetime, optional): Start date for calculations
    """
    # Filter DataFrame by start_date if provided
    if start_date:
        price_df = price_df[price_df.index >= pd.to_datetime(start_date)]
    
    # Initialize DataFrames to store results
    rsi_df = pd.DataFrame()
    drawdown_df = pd.DataFrame()
    max_drawdowns = {}
    
    # Calculate RSI and Drawdown for each asset
    for column in price_df.columns:
        # Calculate 14-day RSI
        rsi_df[column] = ta.rsi(price_df[column], length=14)
        
        # Calculate Drawdown from start_date
        rolling_max = price_df[column].expanding().max()
        drawdown = (price_df[column] - rolling_max) / rolling_max * 100
        drawdown_df[column] = drawdown
        
        # Calculate max historical drawdown from start_date
        max_drawdowns[column] = drawdown.min()
    
    # Get current values
    current_rsi = rsi_df.iloc[-1]
    current_drawdown = drawdown_df.iloc[-1]
    
    # Create DataFrame for plotting
    plot_df = pd.DataFrame({
        'Asset': current_rsi.index,
        'RSI': current_rsi.values,
        'Drawdown': current_drawdown.values,
        'Max_Drawdown': [max_drawdowns[asset] for asset in current_rsi.index]
    })
    
    # Create scatter plot with Drawdown on Y-axis
    fig = px.scatter(
        plot_df,
        x='RSI',
        y='Drawdown',
        text='Asset',
        title=f'Portfolio Assets: Drawdown vs RSI (from {price_df.index[0].strftime("%Y-%m-%d")})',
        labels={
            'RSI': '14-day RSI',
            'Drawdown': 'Current Drawdown (%)'
        }
    )
    
        # Update layout
    fig.update_traces(
        textposition='top center',
        marker=dict(size=8),
        textfont=dict(size=8)  # Add this line to make asset names smaller
    )
    fig.update_layout(
        plot_bgcolor='white',
        showlegend=False,
        hovermode='closest',
        xaxis=dict(
            range=[0, 100],
            gridcolor='lightgrey',
            dtick=10,
            tick0=0,
            tickmode='linear'
        ),
        yaxis=dict(
            gridcolor='lightgrey'
        )
    )
    
    # Add RSI reference lines (vertical)
    fig.add_vline(x=30, line_dash="dash", line_color="green", opacity=0.5)
    fig.add_vline(x=70, line_dash="dash", line_color="red", opacity=0.5)
    
    # Show plot
    fig.show()
    
    return plot_df

# Usage example:
# 

In [35]:
analyze_portfolio_technicals(roster_price_df, start_date='2023-01-01')


,Asset,RSI,Drawdown,Max_Drawdown
0,GMX,45.630253,-77.600048,-81.470662
1,ETHFI,37.228878,-85.009490,-86.256561
2,ALPHA,45.287963,-72.356774,-77.293866
3,INJ,39.626577,-71.117009,-74.613005
4,FTM,53.056877,-61.556357,-74.176015
...,...,...,...,...
115,COMP,43.014635,-52.078028,-60.691147
116,POLIS,39.712985,-85.055184,-87.065133
117,LUNA,41.915875,-87.928592,-89.740427
118,BCH,37.761912,-50.658163,-57.604777


In [42]:
def analyze_asset_ma_distance_3d(price_df, start_date=None):
    """
    Calculates the relative distance of each asset's last price from its moving averages (7-day, 30-day, 90-day)
    and plots a 3D scatter plot where each axis represents one of these distances.
    
    The relative distance is defined as: (last_price - moving_average) / moving_average.
    
    The function also returns breadth measurements, i.e. the percentage of assets with a positive relative distance,
    for each moving average period.
    
    Parameters:
        price_df (pd.DataFrame): DataFrame with price history for portfolio assets.
                                 Columns represent assets; index should be datetime.
        start_date (str or datetime, optional): Only use data from this date onward.
    
    Returns:
        tuple: (fig, output) where fig is a Plotly 3D scatter plot figure and output is a dictionary with:
            - breadth_7d: Percentage of assets with relative distance > 0 for the 7-day MA.
            - breadth_30d: Percentage of assets with relative distance > 0 for the 30-day MA.
            - breadth_90d: Percentage of assets with relative distance > 0 for the 90-day MA.
    """
    import pandas as pd
    import numpy as np
    import plotly.express as px

    # Filter the DataFrame by start_date if provided
    if start_date:
        price_df = price_df[price_df.index >= pd.to_datetime(start_date)]
    
    assets = []
    dist_7d = []
    dist_30d = []
    dist_90d = []
    
    # For each asset, compute the last price and its moving averages using a rolling window.
    # We assume that there is enough data for a 90-day window.
    for asset in price_df.columns:
        series = price_df[asset].dropna()
        if len(series) < 90:  # Skip asset if not enough history
            continue
        
        last_price = series.iloc[-1]
        
        # Compute moving averages
        sma7 = series.rolling(window=90).mean().iloc[-1]
        sma30 = series.rolling(window=180).mean().iloc[-1]
        sma90 = series.rolling(window=365).mean().iloc[-1]
        
        # In case any of the moving averages are nan, skip the asset
        if pd.isna(sma7) or pd.isna(sma30) or pd.isna(sma90):
            continue
        
        # Calculate relative distance: (price - moving average) / moving average
        d7 = (last_price - sma7) / sma7
        d30 = (last_price - sma30) / sma30
        d90 = (last_price - sma90) / sma90
        
        assets.append(asset)
        dist_7d.append(d7)
        dist_30d.append(d30)
        dist_90d.append(d90)
    
    # Create a DataFrame with the calculated distances for plotting.
    distance_df = pd.DataFrame({
        'Asset': assets,
        'dist_7d': dist_7d,
        'dist_30d': dist_30d,
        'dist_90d': dist_90d
    })
    
    # Calculate the breadth for each time frame: the percentage of assets with a positive distance.
    if len(distance_df) > 0:
        breadth_7d = (distance_df['dist_7d'] > 0).mean() * 100
        breadth_30d = (distance_df['dist_30d'] > 0).mean() * 100
        breadth_90d = (distance_df['dist_90d'] > 0).mean() * 100
    else:
        breadth_7d = breadth_30d = breadth_90d = None
    
    output = {
        'breadth_7d': breadth_7d,
        'breadth_30d': breadth_30d,
        'breadth_90d': breadth_90d
    }
    
    # Create a 3D scatter plot of the distances using Plotly Express.
    fig = px.scatter_3d(
        distance_df,
        x='dist_7d',
        y='dist_30d',
        z='dist_90d',
        text='Asset',
        title='Asset Relative Distance from MAs (7d, 30d, 90d)',
        labels={
            'dist_7d': 'Distance from 7d MA',
            'dist_30d': 'Distance from 30d MA',
            'dist_90d': 'Distance from 90d MA'
        }
    )
    
    fig.update_traces(marker=dict(size=5), textposition='top center', textfont=dict(size=10))
    fig.update_layout(scene=dict(
        xaxis_title='Relative Distance from 7d MA',
        yaxis_title='Relative Distance from 30d MA',
        zaxis_title='Relative Distance from 90d MA'
    ))
    
    fig.show()
    
    return fig, output

In [43]:
# Assuming roster_price_df holds your historical price data
fig, breadth_measurements = analyze_asset_ma_distance_3d(roster_price_df, start_date='2023-01-01')
print(breadth_measurements)

{'breadth_7d': 10.833333333333334, 'breadth_30d': 25.833333333333336, 'breadth_90d': 25.0}
